# Operating Systems Internals: A Hands-On Series

**Platform:** Cross-platform (Linux primary, Windows secondary)

---

## 📚 Series Roadmap

| Part | Topic | Focus |
|------|-------|-------|
| **1** | **Foundations** | Processes vs. Threads, System Calls, Privilege Rings |
| 2 | **ELF Deep Dive** | Headers, Sections, Segments, Symbols, Relocations |
| 3 | **PE Deep Dive** | DOS Header, COFF, Sections, Imports/Exports, Resources |
| 4 | **Dynamic Linking** | PLT/GOT, Loaders, `ld.so`, Windows Loader, DLL Injection |
| 5 | **Windows Internals** | PEB, TEB, NTDLL, Native APIs, Handles |
| 6 | **Linux Internals** | VFS, Procfs, Sysfs, Kernel Modules, `ftrace` |

> **Prerequisites:** Basic Python, comfort with hex/byte concepts, and curiosity about what happens *below* `main()`.


## Part 1: What We'll Cover Today

1. **Processes vs. Threads** — The fundamental unit of execution. Memory layout, creation, and the Thread Control Block.
2. **System Calls** — The gateway to the kernel. How userland asks the OS to do the dirty work.
3. **Privilege Rings** — Why your code can't just read the kernel's memory (usually). User mode vs. Kernel mode.

Let's start by importing what we need.


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Imports & Utilities
# ═══════════════════════════════════════════════════════════════

import os
import sys
import threading
import multiprocessing
import time
import ctypes
import struct
import subprocess
import platform

# Platform detection
IS_LINUX = sys.platform.startswith('linux')
IS_WINDOWS = sys.platform == 'win32'

print(f"Platform: {platform.system()} {platform.release()}")
print(f"Architecture: {platform.machine()}")
print(f"Python: {sys.version.split()[0]}")
print(f"PID: {os.getpid()}")

# Utility: run shell command and return output
def shell(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout + result.stderr


## 1. Processes vs. Threads

### The Mental Model

| | **Process** | **Thread** |
|---|---|---|
| **Definition** | An instance of a running program | A single flow of execution *within* a process |
| **Memory** | Owns its own virtual address space | Shares code, heap, and globals; has private stack |
| **Overhead** | High (new page tables, file descriptors, etc.) | Low (just a stack and register set) |
| **Communication** | IPC (pipes, sockets, shared memory) | Direct memory access (shared state) |
| **OS Unit** | Resource allocation | CPU scheduling |

### Memory Layout of a Process


High Addresses 

┌─────────────────┐ 

│   Kernel Space  │ ← Inaccessible from user mode (on most systems) 

│   (shared)      │ 

├─────────────────┤ 

│      Stack      │ ← Grows downward, local variables, return addresses 

│   (per thread)  │ 

├─────────────────┤ 

│  Memory Mapping │ ← mmap'd files, shared libraries, heap expansions 

│      Region     │ 

├─────────────────┤ 

│       Heap      │ ← malloc/new, grows upward 

│     (shared)    │ 

├─────────────────┤ 

│       BSS       │ ← Uninitialized global/static variables 

├─────────────────┤ 

│       Data      │ ← Initialized global/static variables

├─────────────────┤ 

│       Text      │ ← Machine code (read-only) 

│     (shared)    │ 

└─────────────────┘ 

Low Addresses




### Key Insight
When you create a **thread**, the OS creates a new execution context (registers, stack, program counter) but *shares* the address space. When you create a **process**, the OS clones the address space (often copy-on-write) and gives it a fresh set of resources.

> **Linux Trivia:** Linux doesn't really distinguish between processes and threads at the kernel level — both are `task_struct`s. The difference is just which resources they share (`clone()` flags). `fork()` = new process. `pthread_create()` = `clone()` with `CLONE_VM | CLONE_FS | CLONE_FILES | CLONE_SIGHAND`.


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Demonstrating Processes vs. Threads
# ═══════════════════════════════════════════════════════════════

shared_counter = 0  # Global variable in parent process memory

def worker_thread(name):
    """Thread worker: shares memory space with parent"""
    global shared_counter
    shared_counter += 1
    print(f"[THREAD {name}]  PID={os.getpid()}  TID={threading.current_thread().ident}  "
          f"shared_counter={shared_counter}  (same address space)")

def worker_process(name, queue):
    """Process worker: gets its own copy of memory (via fork/spawn)"""
    local_counter = 0  # This is a COPY, not shared
    local_counter += 1
    print(f"[PROCESS {name}] PID={os.getpid()}  PPID={os.getppid()}  "
          f"local_counter={local_counter}  (isolated address space)")
    queue.put(os.getpid())

print("=" * 60)
print("CREATING THREADS")
print("=" * 60)

threads = []
for i in range(3):
    t = threading.Thread(target=worker_thread, args=(f"T-{i}",))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print(f"\n[MAIN THREAD] Final shared_counter = {shared_counter} "
      f"(threads mutated the SAME variable)\n")

print("=" * 60)
print("CREATING PROCESSES")
print("=" * 60)

# Using multiprocessing Queue for safe IPC
from multiprocessing import Queue
q = Queue()
processes = []
for i in range(3):
    p = multiprocessing.Process(target=worker_process, args=(f"P-{i}", q))
    processes.append(p)
    p.start()

child_pids = []
for _ in range(3):
    child_pids.append(q.get())

for p in processes:
    p.join()

print(f"\n[MAIN PROCESS] Child PIDs spawned: {child_pids}")
print(f"[MAIN PROCESS] shared_counter in parent is still {shared_counter} "
      f"(processes did NOT mutate parent's memory)")


## 2. System Calls: The Userland ↔ Kernel Gateway

### What is a System Call?

Your program runs in **user mode** (Ring 3). It cannot:
- Access hardware directly
- Read/write kernel memory
- Change page tables
- Send network packets directly

When it needs these privileges, it executes a **system call** — a controlled trap into **kernel mode** (Ring 0).

### The Syscall Path (Linux x86_64)

User Code

│  
▼ 

mov rax, 0x2c ; syscall number (e.g., 44 = sendto)

mov rdi, fd ; arg1 

mov rsi, buf ; arg2 

mov rdx, len ; arg3 syscall ; CPU enters Ring 0, jumps to syscall table  

│  
▼ 

Kernel: syscall_entry()  

│  
▼ 

sys_call_table[rax] → sys_sendto()  

│  
▼ 

Return to userland (sysret/iret)


### Key Syscall Categories
| Category | Examples (Linux) | What They Do |
|----------|-----------------|--------------|
| Process Control | `fork`, `execve`, `exit`, `wait` | Create/destroy processes |
| File I/O | `open`, `read`, `write`, `close` | File descriptor operations |
| Memory | `mmap`, `munmap`, `mprotect`, `brk` | Virtual memory management |
| Network | `socket`, `bind`, `connect`, `sendto` | Berkeley sockets |
| IPC | `pipe`, `shmget`, `msgget` | Inter-process communication |

### Observing Syscalls: `strace`
The `strace` tool intercepts and prints every system call a process makes. It's the single most useful tool for understanding what a program actually asks the kernel to do.


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Observing System Calls with Python + strace
# ═══════════════════════════════════════════════════════════════

# First, let's see what syscalls Python's "open" actually triggers
print("Let's trace the syscalls made by a simple file open...\n")

if IS_LINUX:
    # Create a small script to trace
    trace_script = '''
import os
fd = os.open("/tmp/os_internals_test.txt", os.O_CREAT | os.O_WRONLY)
os.write(fd, b"Hello from Ring 0")
os.close(fd)
os.unlink("/tmp/os_internals_test.txt")
'''
    with open("/tmp/trace_me.py", "w") as f:
        f.write(trace_script)
    
    # Run it under strace, filtering for key syscalls
    print(shell("strace -e trace=openat,write,close,unlinkat python3 /tmp/trace_me.py 2>&1 | head -30"))
else:
    print("strace is Linux-only. On Windows, use ProcMon from Sysinternals.")

print("\n" + "=" * 60)
print("SYSCALL NUMBERS (Linux x86_64)")
print("=" * 60)

# Common syscall numbers on x86_64 Linux
SYSCALLS = {
    0: "read",
    1: "write",
    2: "open",
    3: "close",
    9: "mmap",
    12: "brk",
    39: "getpid",
    57: "fork",
    59: "execve",
    60: "exit",
    63: "uname",
    102: "getuid",
}

for num, name in SYSCALLS.items():
    print(f"  {num:3d} → {name}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Invoking Syscalls Directly (Raw)
# ═══════════════════════════════════════════════════════════════

# Python's os module wraps syscalls. Let's go one level deeper
# and call libc directly using ctypes to see the raw interface.

libc = ctypes.CDLL(None)  # None loads the standard C library on Unix

# Get the raw syscall function
syscall = libc.syscall
syscall.restype = ctypes.c_long

print("Direct syscall invocation examples:\n")

if IS_LINUX:
    # getpid() syscall = 39 on x86_64
    # Note: os.getpid() is actually cached in Python; this proves it's a syscall
    pid = syscall(39)  # SYS_getpid
    print(f"  syscall(39) [getpid]      = {pid}")
    
    # getuid() syscall = 102 on x86_64
    uid = syscall(102)  # SYS_getuid
    print(f"  syscall(102) [getuid]     = {uid}")
    
    # uname() syscall = 63 on x86_64
    class Utsname(ctypes.Structure):
        _fields_ = [
            ("sysname", ctypes.c_char * 65),
            ("nodename", ctypes.c_char * 65),
            ("release", ctypes.c_char * 65),
            ("version", ctypes.c_char * 65),
            ("machine", ctypes.c_char * 65),
            ("domainname", ctypes.c_char * 65),
        ]
    
    buf = Utsname()
    result = syscall(63, ctypes.byref(buf))  # SYS_uname
    if result == 0:
        print(f"  syscall(63) [uname]       = OK")
        print(f"      sysname:  {buf.sysname.decode()}")
        print(f"      release:  {buf.release.decode()}")
        print(f"      machine:  {buf.machine.decode()}")

elif IS_WINDOWS:
    # On Windows, syscalls go through ntdll.dll (we'll cover this in Part 5)
    print("Windows uses the Native API (ntdll.dll) rather than direct syscalls.")
    print("We'll explore NtCreateFile, NtAllocateVirtualMemory, etc. in Part 5.")

print("\n" + "=" * 60)
print("THE BOUNDARY: Every 'os' module call is a syscall wrapper")
print("=" * 60)
print(f"  os.getpid()  →  syscall(39)  →  kernel's sys_getpid()")
print(f"  os.open()    →  syscall(2)   →  kernel's sys_open()")
print(f"  os.read()    →  syscall(0)   →  kernel's sys_read()")


## 3. Privilege Rings: User Mode vs. Kernel Mode

### The Protection Model

Modern CPUs (x86, ARM, RISC-V) implement **privilege levels** or **rings**:



┌─────────────────────────────────────┐ │ Ring 0 │ Kernel / OS │ ← Full access to hardware, memory, interrupts │ Ring 1 │ Device drivers (rare) │ │ Ring 2 │ Device drivers (rare) │ │ Ring 3 │ User applications │ ← Your code lives here └─────────────────────────────────────┘



Most operating systems use only **Ring 0** (kernel) and **Ring 3** (user).

### What Ring 3 CANNOT Do
1. **Execute privileged instructions**: `cli` (disable interrupts), `hlt` (halt CPU), `in`/`out` (port I/O), `lgdt` (load GDT)
2. **Access kernel memory**: The page tables mark kernel pages as "supervisor only"
3. **Modify page tables**: The `cr3` register is privileged
4. **Access I/O ports**: Unless the I/O bitmap in the TSS allows it

### How the Transition Happens
| Transition | Mechanism | Example |
|------------|-----------|---------|
| User → Kernel | `syscall` / `sysenter` instruction | Your code calls `open()` |
| User → Kernel | Interrupt/exception | Page fault, division by zero |
| User → Kernel | Software interrupt (legacy) | `int 0x80` on old Linux |
| Kernel → User | `sysret` / `iret` | Kernel returns from handling request |

### The Page Table Trick
Each process has page tables. In Linux x86_64:
- **User space**: `0x0000_0000_0000_0000` – `0x0000_7FFF_FFFF_FFFF` (128 TB)
- **Kernel space**: `0xFFFF_8000_0000_0000` – `0xFFFF_FFFF_FFFF_FFFF` (128 TB)

The kernel is mapped into *every* process's address space (so syscalls don't need expensive page table switches), but the pages are marked **supervisor-only**. If Ring 3 code tries to touch them → **Page Fault** → likely **SIGSEGV**.

> **Why map the kernel into every process?** Speed. Syscalls stay in the same address space; only the privilege level changes.


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Detecting Privilege Levels & Capabilities
# ═══════════════════════════════════════════════════════════════

import getpass

print("=" * 60)
print("PRIVILEGE DETECTION")
print("=" * 60)

# Am I root/admin?
if IS_LINUX:
    euid = os.geteuid()
    print(f"  Effective UID (EUID): {euid}")
    print(f"  Running as root: {'YES' if euid == 0 else 'NO'}")
    
    # Linux capabilities (fine-grained privileges)
    cap_data = shell("capsh --print 2>/dev/null || echo 'capsh not installed'")
    if "capsh not installed" not in cap_data:
        print(f"\n  Linux Capabilities:\n{cap_data[:500]}")
    else:
        print("\n  (Install libcap2-bin for capability inspection)")
        
    # Check if we can read /proc/kallsyms (kernel symbols)
    kallsyms = shell("head -5 /proc/kallsyms 2>&1")
    print(f"\n  /proc/kallsyms access (kernel symbol table):")
    for line in kallsyms.strip().split('\n')[:5]:
        print(f"    {line}")

elif IS_WINDOWS:
    import ctypes
    is_admin = ctypes.windll.shell32.IsUserAnAdmin() != 0
    print(f"  Running as Administrator: {'YES' if is_admin else 'NO'}")
    
    # Windows uses Access Tokens instead of UIDs
    print("  Windows uses Access Tokens and Integrity Levels for privilege control.")
    print("  We'll explore tokens in depth in Part 5 (Windows Internals).")

print("\n" + "=" * 60)
print("DEMONSTRATING THE BOUNDARY: Try to touch kernel memory")
print("=" * 60)

# On Linux, we can attempt to read from a kernel address (will segfault)
# We'll do this safely using a subprocess so we don't crash the notebook kernel

if IS_LINUX:
    crash_test = '''
import ctypes
try:
    # 0xFFFF_8880_0000_0000 is a typical kernel virtual address on Linux
    ptr = ctypes.c_void_p(0xFFFF888000000000)
    val = ctypes.c_long.from_address(ptr.value).value
    print("Unexpectedly succeeded!")
except Exception as e:
    print(f"Expected failure: {type(e).__name__}: {e}")
'''
    print("Attempting to read kernel virtual address from child process...")
    result = subprocess.run([sys.executable, '-c', crash_test], 
                          capture_output=True, text=True)
    print(f"  {result.stdout.strip()}")
    print(f"  (The OS protected kernel memory from Ring 3 access)")

print("\n" + "=" * 60)
print("SUMMARY: The syscall is the ONLY legal door from Ring 3 to Ring 0")
print("=" * 60)


## 📋 Part 1 Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Process** | Isolated execution environment with its own memory, FDs, and resources |
| **Thread** | Lightweight execution context sharing the process address space |
| **System Call** | The controlled trap from Ring 3 → Ring 0 to request kernel services |
| **Privilege Rings** | Hardware-enforced isolation. User code cannot touch kernel memory or hardware directly |

### What's Coming in Part 2
**ELF Deep Dive** — We'll write a Python parser to dissect:
- ELF Header (magic, architecture, entry point)
- Program Headers (segments for the loader)
- Section Headers (metadata for the linker)
- The `.text`, `.data`, `.bss`, `.dynsym`, `.plt`, `.got` sections
- How `ld-linux.so` resolves symbols at runtime

> **Exercise before Part 2:** Run `readelf -a /bin/ls` on Linux and try to map each section to the memory layout diagram above.
